# Model Benchmarking — Offline Training & Comparison

Train Random Forest, Ridge, and LSTM models offline and compare their performance
before deploying the training pipeline to CI/CD.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from src.config import settings
from src.training_pipeline.loader import load_feature_view, create_labels, split_data, preprocess
from src.training_pipeline.models import build_random_forest, build_ridge, build_lstm, reshape_for_lstm
from src.training_pipeline.evaluator import evaluate_model, format_metrics_table, compare_models
from src.hopsworks_setup import get_feature_store

print('Imports OK')

## 1. Load Training Data

In [ ]:
fs = get_feature_store()
df, _ = load_feature_view(fs)
df = create_labels(df)
X_train, X_test, y_train, y_test = split_data(df)
X_train_s, X_test_s, scaler = preprocess(X_train, X_test)

feature_names = list(X_train.columns)
horizon_labels = [f'day{i+1}' for i in range(len(settings.prediction_horizons))]

print(f'Train: {X_train_s.shape}, Test: {X_test_s.shape}')
print(f'Features: {feature_names}')

## 2. Train Random Forest

In [ ]:
rf = build_random_forest()
rf.fit(X_train_s, y_train.values)
rf_metrics = evaluate_model(rf, X_test_s, y_test.values, horizon_labels)
print(f'RF — RMSE: {rf_metrics["avg_rmse"]:.2f}, MAE: {rf_metrics["avg_mae"]:.2f}, R²: {rf_metrics["avg_r2"]:.4f}')

## 3. Train Ridge Regression

In [ ]:
ridge = build_ridge()
ridge.fit(X_train_s, y_train.values)
ridge_metrics = evaluate_model(ridge, X_test_s, y_test.values, horizon_labels)
print(f'Ridge — RMSE: {ridge_metrics["avg_rmse"]:.2f}, MAE: {ridge_metrics["avg_mae"]:.2f}, R²: {ridge_metrics["avg_r2"]:.4f}')

## 4. Train LSTM (optional — requires TensorFlow)

In [ ]:
try:
    X_train_seq = reshape_for_lstm(X_train_s)
    X_test_seq = reshape_for_lstm(X_test_s)
    trim = len(X_train_s) - len(X_train_seq)
    
    lstm = build_lstm(n_features=X_train_s.shape[1], n_outputs=len(settings.prediction_horizons))
    lstm.fit(X_train_seq, y_train.values[trim:], epochs=50, batch_size=32,
             validation_split=0.1, verbose=1,
             callbacks=[lstm.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)])
    
    lstm_metrics = evaluate_model(lstm, X_test_seq, y_test.values[len(X_test_s)-len(X_test_seq):], horizon_labels)
    print(f'LSTM — RMSE: {lstm_metrics["avg_rmse"]:.2f}, MAE: {lstm_metrics["avg_mae"]:.2f}, R²: {lstm_metrics["avg_r2"]:.4f}')
except ImportError:
    print('TensorFlow not installed — skipping LSTM')
    lstm_metrics = None

## 5. Model Comparison

In [ ]:
all_results = {'RandomForest': rf_metrics, 'Ridge': ridge_metrics}
if lstm_metrics:
    all_results['LSTM'] = lstm_metrics

print(format_metrics_table(all_results, horizon_labels))
best = compare_models(all_results)
print(f'\nBest model: {best}')

## 6. Actual vs Predicted Visualization

In [ ]:
y_pred_rf = rf.predict(X_test_s)

fig = go.Figure()
for i, label in enumerate(horizon_labels):
    fig.add_trace(go.Scatter(
        x=y_test.values[:, i],
        y=y_pred_rf[:, i],
        mode='markers',
        name=label,
        opacity=0.6,
        marker=dict(size=6)
    ))

# Diagonal reference line
min_val = min(y_test.values.min(), y_pred_rf.min())
max_val = max(y_test.values.max(), y_pred_rf.max())
fig.add_trace(go.Scatter(
    x=[min_val, max_val], y=[min_val, max_val],
    mode='lines', name='Perfect',
    line=dict(dash='dash', color='gray')
))

fig.update_layout(
    title='Actual vs Predicted AQI (Random Forest)',
    xaxis_title='Actual AQI',
    yaxis_title='Predicted AQI',
    height=500
)
fig.show()

## 7. Hyperparameter Tuning Ideas

- **Random Forest**: tune `n_estimators` (100–500), `max_depth` (10–50)
- **Ridge**: tune `alpha` (0.01–100) with cross-validation
- **LSTM**: experiment with 32–128 units, 1–3 layers, different dropout rates
- Try gradient boosting (XGBoost, LightGBM) as additional baselines